# RMSNorm — Fused Single-Pass Triton Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep03-rmsnorm.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series — Episode 3: Just Fuse It — RMSNorm and Softmax. Full write-up: [ep03-rmsnorm-softmax-fused.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep03-rmsnorm-softmax-fused.md).

RMSNorm runs 65 times per Bielik forward pass. It is memory-bound, so this episode fuses the whole normalization into a single-pass Triton kernel (`kernels/normalization/rms_norm_simple.py`) instead of doing it as several separate PyTorch ops.

This notebook lets you run this kernel's correctness check and benchmark on a free Colab GPU — no local NVIDIA hardware required.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works fine).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Correctness check

Compare the Triton kernel output against a plain PyTorch reference implementation.

In [ ]:
import torch
from kernels.normalization.rms_norm_simple import rms_norm_simple

torch.manual_seed(0)
x = torch.randn(8, 1536, device="cuda", dtype=torch.float32)
weight = torch.randn(1536, device="cuda", dtype=torch.float32)

def rms_norm_ref(x, weight, eps=1e-6):
    variance = x.pow(2).mean(-1, keepdim=True)
    return x * torch.rsqrt(variance + eps) * weight

out_triton = rms_norm_simple(x, weight)
out_ref = rms_norm_ref(x, weight)

max_diff = (out_triton - out_ref).abs().max().item()
print(f"max abs diff vs PyTorch reference: {max_diff:.6f}")
assert torch.allclose(out_triton, out_ref, atol=1e-4, rtol=1e-4)
print("Correctness check passed.")

## 2. Benchmark

Runs the same sweep as `make benchmark-rms-norm` and regenerates the plots used in the episode doc. Takes roughly 1-3 minutes on a T4.

In [ ]:
!PYTHONPATH=. python benchmarks/normalization/benchmark_rms_norm.py --save-plots --plot-dir=docs/plots/normalization/

## 3. Results

In [ ]:
from IPython.display import Image, display

plot_dir = "docs/plots/normalization/"
for fname in [
    "rmsnorm-bandwidth-vs-hidden-size.png",
    "rmsnorm-bandwidth-vs-rows.png",
    "rms_norm-summary-bielik-config-gb_s.png",
    "rms_norm-summary-bielik-config-tflops.png",
]:
    display(Image(filename=f"{plot_dir}{fname}"))

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep03-rmsnorm-softmax-fused.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)